# RAM / RAM+ IF-Math Diagnostics (FP32, Analysis-Only)

## Scope (Steps 1-7 Only)
- This notebook only implements the **analysis pipeline** for RAM/RAM+ preparation.
- Included scope:
  - FP32 model loading and validation
  - Paper-formula vs public-code mapping notes
  - Function-by-function explanation of the provided public implementation
  - IF/Math task-vector extraction from a shared base
  - Shared/Unique element analysis (layer-wise + global)
  - Visualization of ratios and delta-magnitude distributions
  - Integrity checks for overlap identities and ratio ranges
- Excluded scope:
  - Unique-only model save (step 8)
  - RAM merge model save (step 9)
  - RAM+ merge model save (step 10)
  - Post-merge reload checks (step 11+)


In [ ]:
from __future__ import annotations

import gc
import json
import random
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, MutableMapping, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for RAM/RAM+ preparatory diagnostics.

    Args:
        base_model_id: Hugging Face model id used as anchor (`theta_base`).
        if_model_path: Local checkpoint path for IF-tuned model.
        math_model_path: Local checkpoint path for Math-tuned model.
        artifact_dir: Directory where CSV/JSON/PNG diagnostics are saved.
        seed: Global seed for deterministic sampling and reproducibility.
        model_dtype: Mandatory floating dtype for all load/analysis operations.
        device: Device used for model loading. Analysis is CPU-based by design.
        threshold: Delta magnitude threshold (`tau`) used for active-coordinate masks.
        dist_sample_cap_per_group: Maximum sampled points per distribution group
            to avoid notebook memory blow-up during histogram visualization.
    """

    base_model_id: str
    if_model_path: Path
    math_model_path: Path
    artifact_dir: Path
    seed: int
    model_dtype: torch.dtype
    device: str
    threshold: float
    dist_sample_cap_per_group: int


RUNTIME = RuntimeConfig(
    base_model_id="Qwen/Qwen3-1.7B",
    if_model_path=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
    ),
    math_model_path=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
    ),
    artifact_dir=Path("merging_analysis/artifacts/ram_if_math_fp32"),
    seed=42,
    model_dtype=torch.float32,
    device="cpu",
    threshold=1e-5,
    dist_sample_cap_per_group=1_000_000,
)

# The requested output filenames are fixed explicitly so downstream scripts can rely on them.
LAYER_STATS_CSV = RUNTIME.artifact_dir / "shared_unique_layer_stats.csv"
GLOBAL_STATS_JSON = RUNTIME.artifact_dir / "shared_unique_global_stats.json"
LAYER_RATIO_PNG = RUNTIME.artifact_dir / "shared_unique_layerwise_ratio.png"
GLOBAL_RATIO_PNG = RUNTIME.artifact_dir / "shared_unique_global_ratio.png"
DELTA_DIST_PNG = RUNTIME.artifact_dir / "delta_magnitude_distribution.png"

# Fail-fast checks make path/dtype problems explicit at notebook start.
if RUNTIME.model_dtype != torch.float32:
    raise ValueError(
        "This notebook must run in strict FP32 mode. "
        f"Configured dtype: {RUNTIME.model_dtype}"
    )

for required_path in [RUNTIME.if_model_path, RUNTIME.math_model_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required checkpoint path does not exist: {required_path}")

RUNTIME.artifact_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")

print(f"Base model id: {RUNTIME.base_model_id}")
print(f"IF model path: {RUNTIME.if_model_path}")
print(f"Math model path: {RUNTIME.math_model_path}")
print(f"Runtime dtype: {RUNTIME.model_dtype}")
print(f"Device: {RUNTIME.device}")
print(f"Threshold (tau): {RUNTIME.threshold}")
print(f"Artifact dir: {RUNTIME.artifact_dir}")


In [ ]:
def set_seed(seed: int) -> None:
    """Set all major random seeds for deterministic behavior.

    Args:
        seed: Integer random seed.

    Returns:
        None. Global RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Write a JSON-serializable mapping to disk with readable formatting.

    Args:
        payload: JSON-serializable mapping object.
        output_path: Destination file path.

    Returns:
        None. The file is persisted to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional `fix_mistral_regex=True` compatibility.

    Why this exists:
    - Some tokenizer classes accept `fix_mistral_regex`; others do not.
    - This fallback keeps the notebook robust across model/tokenizer variants.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.

    Returns:
        Loaded tokenizer instance.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm_fp32(
    model_name_or_path: str | Path,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load a CausalLM checkpoint and tokenizer in strict FP32.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        device: Runtime device (`cpu` expected in this notebook).

    Returns:
        Tuple `(model, tokenizer)` loaded and set to eval mode.
    """

    resolved_path = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved_path,
        torch_dtype=torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    # Explicit recast keeps analysis deterministic even if upstream checkpoint metadata differs.
    model.to(device=device, dtype=torch.float32)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved_path)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def assert_model_float32(model: AutoModelForCausalLM, model_label: str) -> None:
    """Assert that all floating parameters are FP32.

    Args:
        model: Model instance to validate.
        model_label: Human-readable label for error messages.

    Returns:
        None. Raises ValueError when a non-FP32 floating parameter is found.
    """

    for parameter_name, parameter in model.named_parameters():
        if torch.is_floating_point(parameter.data) and parameter.data.dtype != torch.float32:
            raise ValueError(
                "Detected non-FP32 parameter despite strict FP32 requirement | "
                f"model={model_label} | parameter={parameter_name} | dtype={parameter.data.dtype}"
            )


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    task_name: str,
) -> None:
    """Validate named-parameter key and shape compatibility.

    Args:
        base_model: Base model (`theta_base`).
        task_model: Task model aligned to the same architecture.
        task_name: Task identifier used for diagnostics.

    Returns:
        None. Raises ValueError on key/shape mismatch.
    """

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    if set(base_params.keys()) != set(task_params.keys()):
        missing_in_task = sorted(set(base_params.keys()) - set(task_params.keys()))
        missing_in_base = sorted(set(task_params.keys()) - set(base_params.keys()))
        raise ValueError(
            "Named parameter keys mismatch across base/task models. "
            f"task={task_name}, missing_in_task={missing_in_task[:5]}, "
            f"missing_in_base={missing_in_base[:5]}"
        )

    for parameter_name, base_parameter in base_params.items():
        task_parameter = task_params[parameter_name]
        if base_parameter.shape != task_parameter.shape:
            raise ValueError(
                "Parameter shape mismatch across base/task models. "
                f"task={task_name}, parameter={parameter_name}, "
                f"base_shape={tuple(base_parameter.shape)}, "
                f"task_shape={tuple(task_parameter.shape)}"
            )


def parse_layer_name(param_name: str) -> str:
    """Map parameter names to stable layer identifiers.

    This implementation intentionally follows `01_layer_interference_diagnostics.ipynb`
    so layer-level grouping remains directly comparable.

    Args:
        param_name: Full parameter name from `model.named_parameters()`.

    Returns:
        Layer identifier string.
    """

    match = re.search(r"model\.layers\.(\d+)\.", param_name)
    if match:
        layer_index = int(match.group(1))
        return f"layer_{layer_index:02d}"

    if param_name.startswith("model.embed_tokens"):
        return "layer_embed"
    if param_name.startswith("model.norm"):
        return "layer_final_norm"
    if param_name.startswith("lm_head"):
        return "layer_lm_head"
    return "layer_other"


def layer_sort_key(layer_name: str) -> Tuple[int, int, str]:
    """Return sortable key for stable layer ordering.

    Args:
        layer_name: Layer identifier produced by `parse_layer_name`.

    Returns:
        Tuple key that keeps decoder layers in numeric order first.
    """

    match = re.fullmatch(r"layer_(\d+)", layer_name)
    if match:
        return (0, int(match.group(1)), layer_name)

    special_order = {
        "layer_embed": 0,
        "layer_lm_head": 1,
        "layer_final_norm": 2,
        "layer_other": 3,
    }
    return (1, special_order.get(layer_name, 99), layer_name)


def safe_divide(numerator: float, denominator: float) -> float:
    """Safely divide two floats with zero-denominator protection.

    Args:
        numerator: Numerator value.
        denominator: Denominator value.

    Returns:
        `numerator / denominator` when denominator is positive, otherwise `0.0`.
    """

    if denominator <= 0.0:
        return 0.0
    return float(numerator / denominator)


def build_task_vector_from_models(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    task_name: str,
) -> tuple[Dict[str, torch.Tensor], Dict[str, float]]:
    """Build `Delta_task = theta_task - theta_base` in CPU FP32.

    Args:
        base_model: Base model used as anchor.
        task_model: Task model used to compute delta from base.
        task_name: Task identifier used in progress descriptions.

    Returns:
        Tuple `(task_vector, metrics)` where:
        - `task_vector`: mapping `parameter_name -> delta tensor (CPU FP32)`
        - `metrics`: dictionary with aggregate norm and size diagnostics
    """

    validate_parameter_compatibility(
        base_model=base_model,
        task_model=task_model,
        task_name=task_name,
    )

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    task_vector: Dict[str, torch.Tensor] = {}
    l2_norm_sq = 0.0
    total_numel = 0

    with torch.no_grad():
        for parameter_name, base_parameter in tqdm(
            base_params.items(),
            desc=f"Build task vector ({task_name})",
        ):
            # Only floating parameters participate in arithmetic deltas.
            if not torch.is_floating_point(base_parameter):
                continue

            base_fp32_cpu = base_parameter.detach().to(torch.float32).cpu()
            task_fp32_cpu = task_params[parameter_name].detach().to(torch.float32).cpu()
            delta = task_fp32_cpu - base_fp32_cpu

            task_vector[parameter_name] = delta
            l2_norm_sq += float(torch.sum(delta * delta).item())
            total_numel += int(delta.numel())

    metrics = {
        "l2_norm_sq": float(l2_norm_sq),
        "l2_norm": float(np.sqrt(max(l2_norm_sq, 0.0))),
        "numel": int(total_numel),
        "num_tensors": int(len(task_vector)),
    }
    return task_vector, metrics


def append_capped_random_samples(
    target_values: List[float],
    new_values: torch.Tensor,
    cap: int,
    generator: torch.Generator,
) -> None:
    """Append sampled values up to a hard cap for visualization stability.

    Why this helper exists:
    - Full-value collection for billion-parameter models can exceed notebook memory.
    - For distribution plots, a large random sample is typically sufficient.

    Args:
        target_values: Mutable Python list collecting sampled scalars.
        new_values: New tensor values to potentially append.
        cap: Maximum number of values to keep in `target_values`.
        generator: Torch CPU RNG generator for deterministic sampling.

    Returns:
        None. `target_values` is updated in-place.
    """

    if cap <= 0:
        return

    remaining = int(cap - len(target_values))
    if remaining <= 0:
        return

    flat_values = new_values.reshape(-1)
    num_values = int(flat_values.numel())
    if num_values <= 0:
        return

    if num_values <= remaining:
        target_values.extend(flat_values.detach().cpu().tolist())
        return

    # Sample with replacement to avoid large temporary permutations for huge tensors.
    sample_indices = torch.randint(
        low=0,
        high=num_values,
        size=(remaining,),
        generator=generator,
        device=flat_values.device,
    )
    sampled = flat_values[sample_indices]
    target_values.extend(sampled.detach().cpu().tolist())


## RAM / RAM+ Paper Formula vs Public Code Mapping

### RAM (core idea)
- Coordinate-wise, detect whether each task vector changes an element with a threshold mask.
- On active coordinates, merge by averaging active task deltas.
- On inactive coordinates, keep base parameter unchanged.

### RAM+ (core idea)
- Split coordinates by overlap pattern (shared vs non-overlap/unique behavior).
- Preserve overlap merge behavior while re-scaling selected non-overlap components.
- Goal: reduce interference and recover task-specific signals.

### Mapping to Provided Public Code
| Concept | Public Code Function | Practical Behavior |
|---|---|---|
| RAM baseline | `agentic_reinforcement_merge` | Active-coordinate average over task deltas |
| RAM+ variant | `agentic_reinforcement_merge_rescale_v2` (`arm-r-v2`) | Uses overlap statistics to compute per-task rescale for non-overlap updates |
| Extra variants | `arm-r`, `arm-ties`, `arm-unique` | Additional heuristic strategies (not executed in this 1~7 analysis notebook) |

### Important Note on `arm-r-v2`
- `arm-r-v2` is a **code-level heuristic variant** with a specific ratio definition (`overlap / (changed - overlap)`) and clipping behavior.
- This notebook documents and analyzes vectors/masks needed before merge execution, but does **not** run model merge/save.


## Function-by-Function Explanation of Provided Public Code

### `compute_overlap_stats_elementwise_n(base_sd, task_sds, threshold)`
- Computes element-wise change masks (`|theta_task - theta_base| > threshold`) for each task.
- For each task, counts how many changed elements overlap with exactly `k` total active models.
- Outputs per-task totals and overlap percentages.

### `compute_task_vectors(base_sd, task_sds)`
- Constructs task vectors using `Delta_task = theta_task - theta_base` on common keys.
- Returns a list of task-vector dictionaries aligned by parameter name.

### `agentic_reinforcement_merge(base_sd, task_vecs, threshold)`
- RAM-style baseline:
  - active mask on each task vector
  - coordinate-wise sum over active deltas
  - divide by active count (average)
  - add to base

### `agentic_reinforcement_merge_rescale(base_sd, task_vecs, threshold, r)`
- Adds task-wise re-scaling for non-overlap coordinates based on overlap ratio.
- Overlap coordinates still use the standard active average.

### `agentic_reinforcement_merge_rescale_v2(base_sd, task_vecs, threshold, r)`
- Uses alternative overlap ratio formula (`overlap / (changed - overlap)`) and bounded scaling.
- Intended to sharpen non-overlap contribution weighting.

### `agentic_reinforcement_merge_ties(base_sd, task_vecs, threshold)`
- Applies additional pruning/sign-consensus logic over overlap coordinates.
- Uses a TIES-like preservation rule after magnitude pruning.

### `agentic_reinforcement_merge_extract_unique(base_sd, task_vecs, target_index, threshold)`
- Extracts coordinates unique to one chosen task (where exactly one task is active).
- Keeps only target-task unique deltas and applies a fixed scale in provided code.

### `save_merged_model_and_tokenizer(base_path, merged_sd, output_dir)`
- Reloads base architecture in FP32, loads merged state dict, saves model + tokenizer.
- Used after merge completion; intentionally out-of-scope in this notebook.


In [ ]:
set_seed(RUNTIME.seed)

print("Loading base model in FP32...")
base_model, base_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(base_model, model_label="base_model")

# `task_vectors` holds the extracted FP32 deltas used by downstream overlap analysis.
task_vectors: Dict[str, Dict[str, torch.Tensor]] = {}
task_vector_metrics: Dict[str, Dict[str, float]] = {}
task_model_paths: Dict[str, str] = {
    "if": str(RUNTIME.if_model_path),
    "math": str(RUNTIME.math_model_path),
}

for task_name, task_path in [("if", RUNTIME.if_model_path), ("math", RUNTIME.math_model_path)]:
    print(f"Loading task model in FP32 | task={task_name}")
    task_model, _ = load_causal_lm_fp32(
        model_name_or_path=task_path,
        device=RUNTIME.device,
    )
    assert_model_float32(task_model, model_label=f"task_model:{task_name}")

    vector, metrics = build_task_vector_from_models(
        base_model=base_model,
        task_model=task_model,
        task_name=task_name,
    )
    task_vectors[task_name] = vector
    task_vector_metrics[task_name] = metrics

    print(
        f"Task vector stats | task={task_name} | "
        f"num_tensors={metrics['num_tensors']} | "
        f"numel={metrics['numel']} | "
        f"l2_norm={metrics['l2_norm']:.6e}"
    )

    # Free task model immediately to reduce host memory pressure.
    del task_model
    gc.collect()

common_vector_keys = set(task_vectors["if"].keys()) & set(task_vectors["math"].keys())
if not common_vector_keys:
    raise ValueError("No common parameter keys found between IF and Math task vectors.")

if set(task_vectors["if"].keys()) != set(task_vectors["math"].keys()):
    raise ValueError("Task vector key mismatch between IF and Math vectors.")

print(f"Common floating-parameter keys for analysis: {len(common_vector_keys)}")


In [ ]:
def compute_shared_unique_statistics(
    if_task_vector: Mapping[str, torch.Tensor],
    math_task_vector: Mapping[str, torch.Tensor],
    threshold: float,
    sample_cap_per_group: int,
    sample_seed: int,
) -> tuple[pd.DataFrame, Dict[str, Any], Dict[str, List[float]]]:
    """Compute shared/unique element statistics layer-wise and globally.

    Mask definitions:
    - `m_if = |Delta_if| > threshold`
    - `m_math = |Delta_math| > threshold`
    - `shared = m_if & m_math`
    - `unique_if = m_if & ~m_math`
    - `unique_math = m_math & ~m_if`

    Args:
        if_task_vector: IF task-vector mapping.
        math_task_vector: Math task-vector mapping.
        threshold: Active-coordinate threshold (`tau`).
        sample_cap_per_group: Maximum sampled points per distribution group.
        sample_seed: RNG seed for deterministic sampling.

    Returns:
        Tuple `(layer_df, global_stats, magnitude_samples)`.
    """

    common_keys = sorted(set(if_task_vector.keys()) & set(math_task_vector.keys()))
    if not common_keys:
        raise ValueError("No common keys found between IF and Math task vectors.")

    layer_counters: MutableMapping[str, Dict[str, float]] = defaultdict(
        lambda: {
            "num_elements": 0.0,
            "if_changed": 0.0,
            "math_changed": 0.0,
            "shared": 0.0,
            "unique_if": 0.0,
            "unique_math": 0.0,
            "either_changed": 0.0,
        }
    )

    global_counts = {
        "num_elements": 0.0,
        "if_changed": 0.0,
        "math_changed": 0.0,
        "shared": 0.0,
        "unique_if": 0.0,
        "unique_math": 0.0,
        "either_changed": 0.0,
    }

    magnitude_samples: Dict[str, List[float]] = {
        "shared": [],
        "unique_if": [],
        "unique_math": [],
    }
    rng = torch.Generator(device="cpu")
    rng.manual_seed(int(sample_seed))

    with torch.no_grad():
        for parameter_name in tqdm(common_keys, desc="Shared/Unique analysis"):
            if_delta = if_task_vector[parameter_name].detach().to(torch.float32)
            math_delta = math_task_vector[parameter_name].detach().to(torch.float32)

            if if_delta.shape != math_delta.shape:
                raise ValueError(
                    "Shape mismatch between IF and Math deltas. "
                    f"parameter={parameter_name}, "
                    f"if_shape={tuple(if_delta.shape)}, math_shape={tuple(math_delta.shape)}"
                )

            m_if = torch.abs(if_delta) > threshold
            m_math = torch.abs(math_delta) > threshold

            shared_mask = m_if & m_math
            unique_if_mask = m_if & (~m_math)
            unique_math_mask = m_math & (~m_if)
            either_mask = m_if | m_math

            num_elements = float(if_delta.numel())
            if_changed = float(m_if.sum().item())
            math_changed = float(m_math.sum().item())
            shared = float(shared_mask.sum().item())
            unique_if = float(unique_if_mask.sum().item())
            unique_math = float(unique_math_mask.sum().item())
            either_changed = float(either_mask.sum().item())

            layer_name = parse_layer_name(parameter_name)
            layer_counter = layer_counters[layer_name]
            layer_counter["num_elements"] += num_elements
            layer_counter["if_changed"] += if_changed
            layer_counter["math_changed"] += math_changed
            layer_counter["shared"] += shared
            layer_counter["unique_if"] += unique_if
            layer_counter["unique_math"] += unique_math
            layer_counter["either_changed"] += either_changed

            global_counts["num_elements"] += num_elements
            global_counts["if_changed"] += if_changed
            global_counts["math_changed"] += math_changed
            global_counts["shared"] += shared
            global_counts["unique_if"] += unique_if
            global_counts["unique_math"] += unique_math
            global_counts["either_changed"] += either_changed

            # Sample magnitudes by category for distribution plots.
            # We deliberately sample (instead of keeping everything) for memory safety.
            if shared > 0:
                shared_values = torch.maximum(torch.abs(if_delta), torch.abs(math_delta))[shared_mask]
                append_capped_random_samples(
                    target_values=magnitude_samples["shared"],
                    new_values=shared_values,
                    cap=sample_cap_per_group,
                    generator=rng,
                )

            if unique_if > 0:
                unique_if_values = torch.abs(if_delta)[unique_if_mask]
                append_capped_random_samples(
                    target_values=magnitude_samples["unique_if"],
                    new_values=unique_if_values,
                    cap=sample_cap_per_group,
                    generator=rng,
                )

            if unique_math > 0:
                unique_math_values = torch.abs(math_delta)[unique_math_mask]
                append_capped_random_samples(
                    target_values=magnitude_samples["unique_math"],
                    new_values=unique_math_values,
                    cap=sample_cap_per_group,
                    generator=rng,
                )

    rows: List[Dict[str, Any]] = []
    for layer_name in sorted(layer_counters.keys(), key=layer_sort_key):
        counter = layer_counters[layer_name]

        union = counter["shared"] + counter["unique_if"] + counter["unique_math"]
        rows.append(
            {
                "layer": layer_name,
                "num_elements": int(counter["num_elements"]),
                "if_changed": int(counter["if_changed"]),
                "math_changed": int(counter["math_changed"]),
                "shared": int(counter["shared"]),
                "unique_if": int(counter["unique_if"]),
                "unique_math": int(counter["unique_math"]),
                "either_changed": int(counter["either_changed"]),
                "if_changed_ratio": safe_divide(counter["if_changed"], counter["num_elements"]),
                "math_changed_ratio": safe_divide(counter["math_changed"], counter["num_elements"]),
                "shared_ratio_over_all": safe_divide(counter["shared"], counter["num_elements"]),
                "unique_if_ratio_over_all": safe_divide(counter["unique_if"], counter["num_elements"]),
                "unique_math_ratio_over_all": safe_divide(counter["unique_math"], counter["num_elements"]),
                "shared_ratio_over_if_changed": safe_divide(counter["shared"], counter["if_changed"]),
                "shared_ratio_over_math_changed": safe_divide(counter["shared"], counter["math_changed"]),
                "unique_if_ratio_over_if_changed": safe_divide(counter["unique_if"], counter["if_changed"]),
                "unique_math_ratio_over_math_changed": safe_divide(counter["unique_math"], counter["math_changed"]),
                "shared_ratio_over_union": safe_divide(counter["shared"], union),
                "unique_if_ratio_over_union": safe_divide(counter["unique_if"], union),
                "unique_math_ratio_over_union": safe_divide(counter["unique_math"], union),
            }
        )

    layer_df = pd.DataFrame(rows)

    global_union = global_counts["shared"] + global_counts["unique_if"] + global_counts["unique_math"]
    global_stats: Dict[str, Any] = {
        "threshold": float(threshold),
        "num_elements": int(global_counts["num_elements"]),
        "if_changed": int(global_counts["if_changed"]),
        "math_changed": int(global_counts["math_changed"]),
        "shared": int(global_counts["shared"]),
        "unique_if": int(global_counts["unique_if"]),
        "unique_math": int(global_counts["unique_math"]),
        "either_changed": int(global_counts["either_changed"]),
        "if_changed_ratio": safe_divide(global_counts["if_changed"], global_counts["num_elements"]),
        "math_changed_ratio": safe_divide(global_counts["math_changed"], global_counts["num_elements"]),
        "shared_ratio_over_all": safe_divide(global_counts["shared"], global_counts["num_elements"]),
        "unique_if_ratio_over_all": safe_divide(global_counts["unique_if"], global_counts["num_elements"]),
        "unique_math_ratio_over_all": safe_divide(global_counts["unique_math"], global_counts["num_elements"]),
        "shared_ratio_over_if_changed": safe_divide(global_counts["shared"], global_counts["if_changed"]),
        "shared_ratio_over_math_changed": safe_divide(global_counts["shared"], global_counts["math_changed"]),
        "unique_if_ratio_over_if_changed": safe_divide(global_counts["unique_if"], global_counts["if_changed"]),
        "unique_math_ratio_over_math_changed": safe_divide(global_counts["unique_math"], global_counts["math_changed"]),
        "shared_ratio_over_union": safe_divide(global_counts["shared"], global_union),
        "unique_if_ratio_over_union": safe_divide(global_counts["unique_if"], global_union),
        "unique_math_ratio_over_union": safe_divide(global_counts["unique_math"], global_union),
        "nonzero_overlap_jaccard": safe_divide(global_counts["shared"], global_counts["either_changed"]),
        "distribution_sample_sizes": {
            group_name: int(len(values)) for group_name, values in magnitude_samples.items()
        },
    }

    return layer_df, global_stats, magnitude_samples


layer_stats_df, global_stats, delta_magnitude_samples = compute_shared_unique_statistics(
    if_task_vector=task_vectors["if"],
    math_task_vector=task_vectors["math"],
    threshold=float(RUNTIME.threshold),
    sample_cap_per_group=int(RUNTIME.dist_sample_cap_per_group),
    sample_seed=int(RUNTIME.seed),
)

layer_stats_df.to_csv(LAYER_STATS_CSV, index=False)

global_payload = {
    "base_model_id": RUNTIME.base_model_id,
    "if_model_path": str(RUNTIME.if_model_path),
    "math_model_path": str(RUNTIME.math_model_path),
    "task_vector_metrics": task_vector_metrics,
    "global_stats": global_stats,
}
save_json(global_payload, GLOBAL_STATS_JSON)

print(f"Saved layer-wise stats CSV: {LAYER_STATS_CSV}")
print(f"Saved global stats JSON: {GLOBAL_STATS_JSON}")
display(layer_stats_df.head(10))
print(json.dumps(global_stats, indent=2, ensure_ascii=False))


In [ ]:
def compute_tight_ratio_ylim(
    series_list: List[np.ndarray],
    pad_ratio: float = 0.08,
    min_span: float = 0.08,
) -> tuple[float, float]:
    """Compute a tight y-axis range for ratio plots while staying in [0, 1].

    Design rationale:
    - Ratio curves in this notebook usually occupy a narrow band (e.g., 0.15~0.35).
      Forcing [0, 1] suppresses meaningful local variation.
    - This helper computes a data-adaptive range with small padding so trend changes
      remain visually clear without clipping.

    Args:
        series_list: List of 1D ratio arrays used in a subplot.
        pad_ratio: Fractional margin added around observed min/max span.
        min_span: Minimum y-span to avoid over-zoom when values are nearly constant.

    Returns:
        `(y_min, y_max)` clipped to [0, 1] and guaranteed to have non-zero span.
    """

    valid_arrays: List[np.ndarray] = []

    for series in series_list:
        # Convert each input to a flat finite float array so mixed inputs are robustly handled.
        array = np.asarray(series, dtype=np.float64).reshape(-1)
        if array.size == 0:
            continue
        finite_array = array[np.isfinite(array)]
        if finite_array.size == 0:
            continue
        valid_arrays.append(finite_array)

    # Fallback to full ratio range if no valid data is available.
    if not valid_arrays:
        return (0.0, 1.0)

    merged = np.concatenate(valid_arrays, axis=0)
    observed_min = float(np.min(merged))
    observed_max = float(np.max(merged))
    observed_span = observed_max - observed_min

    if observed_span < float(min_span):
        # When values are almost constant, enforce a minimum span around the center.
        center = 0.5 * (observed_min + observed_max)
        half_span = 0.5 * float(min_span)
        lower = center - half_span
        upper = center + half_span
    else:
        padding = observed_span * float(pad_ratio)
        lower = observed_min - padding
        upper = observed_max + padding

    # Ratio metrics are bounded in [0, 1], so clip limits accordingly.
    lower = max(0.0, lower)
    upper = min(1.0, upper)

    # Ensure numerical safety for extremely degenerate edge cases.
    if (upper - lower) < 1e-6:
        lower = max(0.0, lower - 0.05)
        upper = min(1.0, upper + 0.05)

    if (upper - lower) < 1e-6:
        return (0.0, 1.0)

    return (float(lower), float(upper))


def visualize_layerwise_ratios(layer_df: pd.DataFrame, output_path: Path) -> None:
    """Create and save layer-wise shared/unique ratio visualization.

    Args:
        layer_df: Dataframe produced by `compute_shared_unique_statistics`.
        output_path: PNG output path.

    Returns:
        None. Figure is written to disk.
    """

    if layer_df.empty:
        raise ValueError("layer_df is empty. Cannot draw layer-wise ratio figure.")

    plot_df = layer_df.copy().reset_index(drop=True)
    plot_df["plot_index"] = np.arange(len(plot_df))

    fig, axes = plt.subplots(2, 1, figsize=(22, 12), constrained_layout=True)

    # Top panel: ratio over all coordinates per layer.
    axes[0].plot(plot_df["plot_index"], plot_df["shared_ratio_over_all"], marker="o", label="shared / all")
    axes[0].plot(plot_df["plot_index"], plot_df["unique_if_ratio_over_all"], marker="o", label="unique_if / all")
    axes[0].plot(plot_df["plot_index"], plot_df["unique_math_ratio_over_all"], marker="o", label="unique_math / all")
    top_ylim = compute_tight_ratio_ylim(
        series_list=[
            plot_df["shared_ratio_over_all"].to_numpy(dtype=np.float64),
            plot_df["unique_if_ratio_over_all"].to_numpy(dtype=np.float64),
            plot_df["unique_math_ratio_over_all"].to_numpy(dtype=np.float64),
        ],
        pad_ratio=0.10,
        min_span=0.06,
    )
    axes[0].set_ylim(*top_ylim)
    axes[0].set_title("Layer-Wise Shared/Unique Ratios (over all elements)")
    axes[0].set_xlabel("Layer index")
    axes[0].set_ylabel("Ratio")
    axes[0].legend(loc="upper right")

    # Bottom panel: ratio over the active union to highlight competition among active updates.
    axes[1].plot(plot_df["plot_index"], plot_df["shared_ratio_over_union"], marker="o", label="shared / union")
    axes[1].plot(plot_df["plot_index"], plot_df["unique_if_ratio_over_union"], marker="o", label="unique_if / union")
    axes[1].plot(plot_df["plot_index"], plot_df["unique_math_ratio_over_union"], marker="o", label="unique_math / union")
    bottom_ylim = compute_tight_ratio_ylim(
        series_list=[
            plot_df["shared_ratio_over_union"].to_numpy(dtype=np.float64),
            plot_df["unique_if_ratio_over_union"].to_numpy(dtype=np.float64),
            plot_df["unique_math_ratio_over_union"].to_numpy(dtype=np.float64),
        ],
        pad_ratio=0.10,
        min_span=0.06,
    )
    axes[1].set_ylim(*bottom_ylim)
    axes[1].set_title("Layer-Wise Shared/Unique Ratios (over active union)")
    axes[1].set_xlabel("Layer index")
    axes[1].set_ylabel("Ratio")
    axes[1].legend(loc="upper right")

    tick_step = max(1, int(len(plot_df) / 24))
    tick_positions = plot_df["plot_index"][::tick_step]
    tick_labels = plot_df["layer"][::tick_step]

    for axis in axes:
        axis.set_xticks(tick_positions)
        axis.set_xticklabels(tick_labels, rotation=45, ha="right")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved layer-wise ratio figure: {output_path}")


def visualize_global_ratios(global_stat_dict: Mapping[str, Any], output_path: Path) -> None:
    """Create and save global shared/unique ratio visualization.

    Args:
        global_stat_dict: Global stats dictionary.
        output_path: PNG output path.

    Returns:
        None. Figure is written to disk.
    """

    categories = ["shared", "unique_if", "unique_math"]
    counts = [int(global_stat_dict[key]) for key in categories]
    ratios_over_all = [float(global_stat_dict[f"{key}_ratio_over_all"]) for key in categories]
    ratios_over_union = [float(global_stat_dict[f"{key}_ratio_over_union"]) for key in categories]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    sns.barplot(x=categories, y=counts, ax=axes[0], palette="Set2")
    axes[0].set_title("Global Shared/Unique Counts")
    axes[0].set_xlabel("Category")
    axes[0].set_ylabel("Count")

    width = 0.35
    x = np.arange(len(categories))
    axes[1].bar(x - width / 2, ratios_over_all, width=width, label="ratio over all")
    axes[1].bar(x + width / 2, ratios_over_union, width=width, label="ratio over union")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(categories)

    # Keep the global bar chart tight as well so small ratio differences remain visible.
    global_ratio_ylim = compute_tight_ratio_ylim(
        series_list=[np.asarray(ratios_over_all), np.asarray(ratios_over_union)],
        pad_ratio=0.12,
        min_span=0.05,
    )
    axes[1].set_ylim(*global_ratio_ylim)

    axes[1].set_title("Global Shared/Unique Ratios")
    axes[1].set_xlabel("Category")
    axes[1].set_ylabel("Ratio")
    axes[1].legend(loc="upper right")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved global ratio figure: {output_path}")


def visualize_delta_magnitude_distribution(
    sampled_values: Mapping[str, List[float]],
    output_path: Path,
) -> None:
    """Create and save delta-magnitude distribution plot (log scale).

    Args:
        sampled_values: Mapping from category name to sampled absolute delta values.
        output_path: PNG output path.

    Returns:
        None. Figure is written to disk.
    """

    fig, ax = plt.subplots(1, 1, figsize=(12, 6), constrained_layout=True)

    plotted_any = False
    for group_name, values in sampled_values.items():
        if not values:
            continue

        values_array = np.asarray(values, dtype=np.float64)
        # Use log10 transform with epsilon so tiny magnitudes remain representable.
        log_values = np.log10(np.maximum(values_array, 1e-20))
        sns.histplot(
            log_values,
            bins=120,
            stat="density",
            element="step",
            fill=False,
            linewidth=1.4,
            label=f"{group_name} (n={len(values)})",
            ax=ax,
        )
        plotted_any = True

    if not plotted_any:
        raise ValueError("No sampled values available to plot delta magnitude distribution.")

    ax.set_title("Delta Magnitude Distribution by Category (log10 abs delta)")
    ax.set_xlabel("log10(abs(delta))")
    ax.set_ylabel("Density")
    ax.legend(loc="upper left")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved delta-magnitude distribution figure: {output_path}")


visualize_layerwise_ratios(layer_df=layer_stats_df, output_path=LAYER_RATIO_PNG)
visualize_global_ratios(global_stat_dict=global_stats, output_path=GLOBAL_RATIO_PNG)
visualize_delta_magnitude_distribution(sampled_values=delta_magnitude_samples, output_path=DELTA_DIST_PNG)


In [ ]:
# ------------------------------
# Integrity and output checks
# ------------------------------
if layer_stats_df.empty:
    raise ValueError("Layer statistics dataframe is empty.")

required_identity_columns = [
    "shared",
    "unique_if",
    "unique_math",
    "if_changed",
    "math_changed",
]
for column_name in required_identity_columns:
    if column_name not in layer_stats_df.columns:
        raise KeyError(f"Missing required column in layer_stats_df: {column_name}")

# Identity checks requested in the spec.
if not np.all((layer_stats_df["shared"] + layer_stats_df["unique_if"]).values == layer_stats_df["if_changed"].values):
    raise AssertionError("Identity check failed: shared + unique_if != if_changed for some layers.")

if not np.all((layer_stats_df["shared"] + layer_stats_df["unique_math"]).values == layer_stats_df["math_changed"].values):
    raise AssertionError("Identity check failed: shared + unique_math != math_changed for some layers.")

# Global identities should also hold exactly.
if int(global_stats["shared"]) + int(global_stats["unique_if"]) != int(global_stats["if_changed"]):
    raise AssertionError("Global identity failed: shared + unique_if != if_changed")
if int(global_stats["shared"]) + int(global_stats["unique_math"]) != int(global_stats["math_changed"]):
    raise AssertionError("Global identity failed: shared + unique_math != math_changed")

# Ratio bounds check with tolerance.
ratio_columns = [
    "if_changed_ratio",
    "math_changed_ratio",
    "shared_ratio_over_all",
    "unique_if_ratio_over_all",
    "unique_math_ratio_over_all",
    "shared_ratio_over_if_changed",
    "shared_ratio_over_math_changed",
    "unique_if_ratio_over_if_changed",
    "unique_math_ratio_over_math_changed",
    "shared_ratio_over_union",
    "unique_if_ratio_over_union",
    "unique_math_ratio_over_union",
]

for ratio_column in ratio_columns:
    values = layer_stats_df[ratio_column].to_numpy(dtype=np.float64)
    if np.any(values < -1e-8) or np.any(values > 1.0 + 1e-8):
        raise AssertionError(f"Ratio out of [0, 1] range in layer column: {ratio_column}")


# Additional partition checks for union-based interpretation.
# If a coordinate is active in either task, it must belong to exactly one of:
# {shared, unique_if, unique_math}. Therefore:
#   shared + unique_if + unique_math == either_changed
if not np.all(
    (
        layer_stats_df["shared"]
        + layer_stats_df["unique_if"]
        + layer_stats_df["unique_math"]
    ).values
    == layer_stats_df["either_changed"].values
):
    raise AssertionError(
        "Partition identity failed: shared + unique_if + unique_math != either_changed for some layers."
    )

# For layers with union>0, normalized union ratios must sum to 1.
layer_union_count = (
    layer_stats_df["shared"]
    + layer_stats_df["unique_if"]
    + layer_stats_df["unique_math"]
).to_numpy(dtype=np.float64)
layer_union_ratio_sum = (
    layer_stats_df["shared_ratio_over_union"]
    + layer_stats_df["unique_if_ratio_over_union"]
    + layer_stats_df["unique_math_ratio_over_union"]
).to_numpy(dtype=np.float64)

layer_union_active_mask = layer_union_count > 0
if np.any(layer_union_active_mask):
    max_union_sum_error = float(
        np.max(np.abs(layer_union_ratio_sum[layer_union_active_mask] - 1.0))
    )
    if max_union_sum_error > 1e-8:
        raise AssertionError(
            "Union ratio sum check failed: "
            f"max_error={max_union_sum_error:.3e} on active-union layers"
        )
else:
    max_union_sum_error = 0.0

# Global partition identity and union-ratio sum checks.
if (
    int(global_stats["shared"])
    + int(global_stats["unique_if"])
    + int(global_stats["unique_math"])
    != int(global_stats["either_changed"])
):
    raise AssertionError(
        "Global partition identity failed: shared + unique_if + unique_math != either_changed"
    )

global_union_ratio_sum = (
    float(global_stats["shared_ratio_over_union"])
    + float(global_stats["unique_if_ratio_over_union"])
    + float(global_stats["unique_math_ratio_over_union"])
)
global_union_count = (
    int(global_stats["shared"])
    + int(global_stats["unique_if"])
    + int(global_stats["unique_math"])
)

if global_union_count > 0 and abs(global_union_ratio_sum - 1.0) > 1e-8:
    raise AssertionError(
        "Global union ratio sum check failed: "
        f"sum={global_union_ratio_sum:.12f}"
    )

# Emit diagnostics so users can directly confirm ratio-partition consistency.
zero_union_layers = int(np.sum(~layer_union_active_mask))
print(
    "Union-ratio diagnostics | "
    f"active_layers={int(np.sum(layer_union_active_mask))}, "
    f"zero_union_layers={zero_union_layers}, "
    f"max_layer_union_sum_error={max_union_sum_error:.3e}, "
    f"global_union_ratio_sum={global_union_ratio_sum:.12f}"
)

for ratio_key in [
    "if_changed_ratio",
    "math_changed_ratio",
    "shared_ratio_over_all",
    "unique_if_ratio_over_all",
    "unique_math_ratio_over_all",
    "shared_ratio_over_if_changed",
    "shared_ratio_over_math_changed",
    "unique_if_ratio_over_if_changed",
    "unique_math_ratio_over_math_changed",
    "shared_ratio_over_union",
    "unique_if_ratio_over_union",
    "unique_math_ratio_over_union",
    "nonzero_overlap_jaccard",
]:
    value = float(global_stats[ratio_key])
    if value < -1e-8 or value > 1.0 + 1e-8:
        raise AssertionError(f"Ratio out of [0, 1] range in global stats: {ratio_key}")

required_outputs = [
    LAYER_STATS_CSV,
    GLOBAL_STATS_JSON,
    LAYER_RATIO_PNG,
    GLOBAL_RATIO_PNG,
    DELTA_DIST_PNG,
]
for output_path in required_outputs:
    if not output_path.exists():
        raise FileNotFoundError(f"Expected output file does not exist: {output_path}")

print("All integrity checks passed.")
print("Output files:")
for output_path in required_outputs:
    print(f"- {output_path}")

# We intentionally keep task vectors in memory here because later cells
# perform requested FP32 model reconstruction/merge saves using these vectors.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## IF Unique vs Top-k(20/30%) Overlap Analysis (Layer-wise + Global + Distribution)

This section compares **IF unique mask** with IF task-vector top-k masks under two ranking rules:
- `mode="abs"`: top-k by `|Delta_if|`
- `mode="value"`: top-k by raw `Delta_if` value (largest positive values)

Top-k percentages:
- `20%`
- `30%`

Outputs include:
- layer-wise overlap metrics
- global overlap metrics
- per-config distribution plots for abs-delta in overlap/unique-only/topk-only groups


In [ ]:
IF_UNIQUE_TOPK_ANALYSIS_DIR = RUNTIME.artifact_dir / "if_unique_vs_topk_overlap"
IF_UNIQUE_TOPK_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

IF_UNIQUE_TOPK_LAYERWISE_CSV = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_layerwise_metrics.csv"
IF_UNIQUE_TOPK_GLOBAL_CSV = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_global_metrics.csv"
IF_UNIQUE_TOPK_GLOBAL_JSON = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_global_metrics.json"
IF_UNIQUE_TOPK_LAYERWISE_PNG = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_layerwise_metrics.png"
IF_UNIQUE_TOPK_GLOBAL_PNG = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_global_metrics.png"
IF_UNIQUE_TOPK_DIST_PNG = IF_UNIQUE_TOPK_ANALYSIS_DIR / "if_unique_vs_topk_abs_distribution.png"

TOPK_COMPARISON_CONFIGS: List[Dict[str, Any]] = [
    {"mode": "abs", "top_pct": 0.20, "name": "abs_top20"},
    {"mode": "abs", "top_pct": 0.30, "name": "abs_top30"},
    {"mode": "value", "top_pct": 0.20, "name": "value_top20"},
    {"mode": "value", "top_pct": 0.30, "name": "value_top30"},
]

# Cap sampled points per group/config to keep distribution plots memory-safe.
TOPK_DISTRIBUTION_SAMPLE_CAP_PER_GROUP = 400_000


def build_topk_mask_for_tensor(
    score_tensor: torch.Tensor,
    top_pct: float,
) -> tuple[torch.Tensor, int]:
    """Build an exact top-k mask on a single tensor.

    Design choice:
    - Top-k is applied per parameter tensor (not full-model global sort) for memory
      stability on billion-parameter checkpoints.
    - This still yields consistent layer/global overlap diagnostics after aggregation.

    Args:
        score_tensor: Tensor of ranking scores.
        top_pct: Fraction in [0, 1] indicating selected top-k percentage.

    Returns:
        Tuple `(topk_mask, k)` where:
        - `topk_mask` is a boolean tensor with exactly `k` true entries.
        - `k` is the selected count.
    """

    if top_pct < 0.0 or top_pct > 1.0:
        raise ValueError(f"top_pct must be in [0,1], got: {top_pct}")

    flat_scores = score_tensor.reshape(-1)
    numel = int(flat_scores.numel())

    if numel == 0:
        return torch.zeros_like(score_tensor, dtype=torch.bool), 0

    k = int(numel * float(top_pct))
    if top_pct > 0.0 and k == 0:
        k = 1

    if k <= 0:
        return torch.zeros_like(score_tensor, dtype=torch.bool), 0

    if k >= numel:
        return torch.ones_like(score_tensor, dtype=torch.bool), numel

    # `topk` gives exact largest-k indices for this tensor.
    topk_indices = torch.topk(flat_scores, k=k, largest=True, sorted=False).indices

    mask_flat = torch.zeros(numel, dtype=torch.bool, device=flat_scores.device)
    mask_flat[topk_indices] = True
    return mask_flat.view_as(score_tensor), int(k)


def compute_if_unique_vs_topk_overlap(
    if_task_vector: Mapping[str, torch.Tensor],
    math_task_vector: Mapping[str, torch.Tensor],
    threshold: float,
    config_list: List[Mapping[str, Any]],
    sample_cap_per_group: int,
    sample_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, List[float]]]]:
    """Compare IF-unique mask with IF top-k masks under multiple ranking configs.

    Unique mask definition:
    - `unique_if = (|Delta_if| > tau) & ~(|Delta_math| > tau)`

    Top-k mask definitions:
    - `mode='abs'`: score is `|Delta_if|`
    - `mode='value'`: score is raw `Delta_if` value (largest positive values)

    Args:
        if_task_vector: IF task-vector mapping (`Delta_if`).
        math_task_vector: Math task-vector mapping (`Delta_math`).
        threshold: Activity threshold `tau` for unique-mask construction.
        config_list: List of top-k configuration dictionaries.
        sample_cap_per_group: Maximum sampled points per config/group for distribution plots.
        sample_seed: RNG seed for deterministic sampling.

    Returns:
        Tuple of:
        - layer-wise metrics dataframe
        - global metrics dataframe
        - sampled abs-delta dictionary for distribution visualization
    """

    common_keys = sorted(set(if_task_vector.keys()) & set(math_task_vector.keys()))
    if not common_keys:
        raise ValueError("No common keys found between IF and Math task vectors.")

    # Nested counters keyed by [config_name][layer_name].
    layer_counters: Dict[str, Dict[str, Dict[str, float]]] = {
        cfg["name"]: defaultdict(
            lambda: {
                "num_elements": 0.0,
                "unique_if_count": 0.0,
                "topk_count": 0.0,
                "overlap_count": 0.0,
                "union_count": 0.0,
                "unique_only_count": 0.0,
                "topk_only_count": 0.0,
            }
        )
        for cfg in config_list
    }

    global_counters: Dict[str, Dict[str, float]] = {
        cfg["name"]: {
            "num_elements": 0.0,
            "unique_if_count": 0.0,
            "topk_count": 0.0,
            "overlap_count": 0.0,
            "union_count": 0.0,
            "unique_only_count": 0.0,
            "topk_only_count": 0.0,
        }
        for cfg in config_list
    }

    # Keep distribution samples per config and subset relation.
    sampled_abs_delta_by_config: Dict[str, Dict[str, List[float]]] = {
        cfg["name"]: {
            "overlap": [],
            "unique_only": [],
            "topk_only": [],
        }
        for cfg in config_list
    }

    rng = torch.Generator(device="cpu")
    rng.manual_seed(int(sample_seed))

    with torch.no_grad():
        for parameter_name in tqdm(common_keys, desc="IF unique vs top-k overlap"):
            if_delta = if_task_vector[parameter_name].detach().to(torch.float32)
            math_delta = math_task_vector[parameter_name].detach().to(torch.float32)

            if if_delta.shape != math_delta.shape:
                raise ValueError(
                    "Shape mismatch between IF and Math deltas during top-k overlap analysis. "
                    f"parameter={parameter_name}"
                )

            # Base unique mask for IF task.
            unique_if_mask = (torch.abs(if_delta) > float(threshold)) & (
                ~(torch.abs(math_delta) > float(threshold))
            )
            unique_if_count = float(unique_if_mask.sum().item())
            num_elements = float(if_delta.numel())
            layer_name = parse_layer_name(parameter_name)

            abs_if_delta = torch.abs(if_delta)

            for cfg in config_list:
                cfg_name = str(cfg["name"])
                mode = str(cfg["mode"])
                top_pct = float(cfg["top_pct"])

                if mode == "abs":
                    score_tensor = abs_if_delta
                elif mode == "value":
                    score_tensor = if_delta
                else:
                    raise ValueError(f"Unsupported top-k mode: {mode}")

                topk_mask, _ = build_topk_mask_for_tensor(score_tensor=score_tensor, top_pct=top_pct)

                overlap_mask = unique_if_mask & topk_mask
                unique_only_mask = unique_if_mask & (~topk_mask)
                topk_only_mask = topk_mask & (~unique_if_mask)
                union_mask = unique_if_mask | topk_mask

                topk_count = float(topk_mask.sum().item())
                overlap_count = float(overlap_mask.sum().item())
                unique_only_count = float(unique_only_mask.sum().item())
                topk_only_count = float(topk_only_mask.sum().item())
                union_count = float(union_mask.sum().item())

                # Layer-level accumulation.
                layer_counter = layer_counters[cfg_name][layer_name]
                layer_counter["num_elements"] += num_elements
                layer_counter["unique_if_count"] += unique_if_count
                layer_counter["topk_count"] += topk_count
                layer_counter["overlap_count"] += overlap_count
                layer_counter["union_count"] += union_count
                layer_counter["unique_only_count"] += unique_only_count
                layer_counter["topk_only_count"] += topk_only_count

                # Global accumulation.
                global_counter = global_counters[cfg_name]
                global_counter["num_elements"] += num_elements
                global_counter["unique_if_count"] += unique_if_count
                global_counter["topk_count"] += topk_count
                global_counter["overlap_count"] += overlap_count
                global_counter["union_count"] += union_count
                global_counter["unique_only_count"] += unique_only_count
                global_counter["topk_only_count"] += topk_only_count

                # Distribution sampling from abs(IF delta) for three relation groups.
                if overlap_count > 0:
                    append_capped_random_samples(
                        target_values=sampled_abs_delta_by_config[cfg_name]["overlap"],
                        new_values=abs_if_delta[overlap_mask],
                        cap=sample_cap_per_group,
                        generator=rng,
                    )

                if unique_only_count > 0:
                    append_capped_random_samples(
                        target_values=sampled_abs_delta_by_config[cfg_name]["unique_only"],
                        new_values=abs_if_delta[unique_only_mask],
                        cap=sample_cap_per_group,
                        generator=rng,
                    )

                if topk_only_count > 0:
                    append_capped_random_samples(
                        target_values=sampled_abs_delta_by_config[cfg_name]["topk_only"],
                        new_values=abs_if_delta[topk_only_mask],
                        cap=sample_cap_per_group,
                        generator=rng,
                    )

    # Build layer-wise dataframe.
    layer_rows: List[Dict[str, Any]] = []
    for cfg in config_list:
        cfg_name = str(cfg["name"])
        mode = str(cfg["mode"])
        top_pct = float(cfg["top_pct"])

        for layer_name in sorted(layer_counters[cfg_name].keys(), key=layer_sort_key):
            counter = layer_counters[cfg_name][layer_name]

            unique_count = float(counter["unique_if_count"])
            topk_count = float(counter["topk_count"])
            overlap_count = float(counter["overlap_count"])
            union_count = float(counter["union_count"])
            num_elements = float(counter["num_elements"])

            precision_over_topk = safe_divide(overlap_count, topk_count)
            recall_over_unique = safe_divide(overlap_count, unique_count)
            jaccard = safe_divide(overlap_count, union_count)

            unique_ratio_over_all = safe_divide(unique_count, num_elements)
            topk_ratio_over_all = safe_divide(topk_count, num_elements)
            overlap_ratio_over_all = safe_divide(overlap_count, num_elements)

            # Random baseline precision is unique density; enrichment >1 implies better-than-random overlap.
            enrichment_over_random = safe_divide(
                precision_over_topk,
                unique_ratio_over_all,
            )

            layer_rows.append(
                {
                    "config_name": cfg_name,
                    "mode": mode,
                    "top_pct": top_pct,
                    "layer": layer_name,
                    "num_elements": int(num_elements),
                    "unique_if_count": int(unique_count),
                    "topk_count": int(topk_count),
                    "overlap_count": int(overlap_count),
                    "union_count": int(union_count),
                    "unique_only_count": int(counter["unique_only_count"]),
                    "topk_only_count": int(counter["topk_only_count"]),
                    "unique_ratio_over_all": unique_ratio_over_all,
                    "topk_ratio_over_all": topk_ratio_over_all,
                    "overlap_ratio_over_all": overlap_ratio_over_all,
                    "precision_over_topk": precision_over_topk,
                    "recall_over_unique": recall_over_unique,
                    "jaccard": jaccard,
                    "enrichment_over_random": enrichment_over_random,
                }
            )

    layerwise_df = pd.DataFrame(layer_rows)

    # Build global dataframe.
    global_rows: List[Dict[str, Any]] = []
    for cfg in config_list:
        cfg_name = str(cfg["name"])
        mode = str(cfg["mode"])
        top_pct = float(cfg["top_pct"])

        counter = global_counters[cfg_name]
        unique_count = float(counter["unique_if_count"])
        topk_count = float(counter["topk_count"])
        overlap_count = float(counter["overlap_count"])
        union_count = float(counter["union_count"])
        num_elements = float(counter["num_elements"])

        precision_over_topk = safe_divide(overlap_count, topk_count)
        recall_over_unique = safe_divide(overlap_count, unique_count)
        jaccard = safe_divide(overlap_count, union_count)

        unique_ratio_over_all = safe_divide(unique_count, num_elements)
        topk_ratio_over_all = safe_divide(topk_count, num_elements)
        overlap_ratio_over_all = safe_divide(overlap_count, num_elements)

        enrichment_over_random = safe_divide(
            precision_over_topk,
            unique_ratio_over_all,
        )

        global_rows.append(
            {
                "config_name": cfg_name,
                "mode": mode,
                "top_pct": top_pct,
                "num_elements": int(num_elements),
                "unique_if_count": int(unique_count),
                "topk_count": int(topk_count),
                "overlap_count": int(overlap_count),
                "union_count": int(union_count),
                "unique_only_count": int(counter["unique_only_count"]),
                "topk_only_count": int(counter["topk_only_count"]),
                "unique_ratio_over_all": unique_ratio_over_all,
                "topk_ratio_over_all": topk_ratio_over_all,
                "overlap_ratio_over_all": overlap_ratio_over_all,
                "precision_over_topk": precision_over_topk,
                "recall_over_unique": recall_over_unique,
                "jaccard": jaccard,
                "enrichment_over_random": enrichment_over_random,
                "distribution_sample_sizes": {
                    group_name: int(len(values))
                    for group_name, values in sampled_abs_delta_by_config[cfg_name].items()
                },
            }
        )

    global_df = pd.DataFrame(global_rows)
    return layerwise_df, global_df, sampled_abs_delta_by_config


def visualize_if_unique_topk_layerwise(
    layerwise_df: pd.DataFrame,
    output_path: Path,
) -> None:
    """Visualize layer-wise overlap metrics across top-k configs.

    Args:
        layerwise_df: Layer-wise overlap dataframe.
        output_path: PNG file path.

    Returns:
        None. Figure is saved to disk.
    """

    if layerwise_df.empty:
        raise ValueError("layerwise_df is empty; cannot visualize layer-wise overlap metrics.")

    metric_specs = [
        ("recall_over_unique", "Recall = overlap / unique_if"),
        ("precision_over_topk", "Precision = overlap / topk"),
        ("jaccard", "Jaccard = overlap / union"),
    ]

    layer_order = sorted(layerwise_df["layer"].unique().tolist(), key=layer_sort_key)
    layer_to_index = {layer_name: index for index, layer_name in enumerate(layer_order)}

    fig, axes = plt.subplots(len(metric_specs), 1, figsize=(24, 15), constrained_layout=True)

    if len(metric_specs) == 1:
        axes = [axes]

    for metric_axis, (metric_key, metric_label) in zip(axes, metric_specs):
        series_collection: List[np.ndarray] = []

        for config_name in sorted(layerwise_df["config_name"].unique().tolist()):
            subset = layerwise_df[layerwise_df["config_name"] == config_name].copy()
            subset["layer_index"] = subset["layer"].map(layer_to_index)
            subset = subset.sort_values("layer_index")

            y_values = subset[metric_key].to_numpy(dtype=np.float64)
            x_values = subset["layer_index"].to_numpy(dtype=np.int64)
            series_collection.append(y_values)

            metric_axis.plot(
                x_values,
                y_values,
                marker="o",
                linewidth=1.8,
                label=config_name,
            )

        metric_ylim = compute_tight_ratio_ylim(
            series_list=series_collection,
            pad_ratio=0.10,
            min_span=0.08,
        )
        metric_axis.set_ylim(*metric_ylim)
        metric_axis.set_title(f"Layer-wise {metric_label}")
        metric_axis.set_xlabel("Layer index")
        metric_axis.set_ylabel("Ratio")
        metric_axis.legend(loc="upper right")

        tick_step = max(1, int(len(layer_order) / 24))
        tick_positions = np.arange(len(layer_order))[::tick_step]
        tick_labels = [layer_order[pos] for pos in tick_positions]
        metric_axis.set_xticks(tick_positions)
        metric_axis.set_xticklabels(tick_labels, rotation=45, ha="right")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved layer-wise overlap figure: {output_path}")


def visualize_if_unique_topk_global(
    global_df: pd.DataFrame,
    output_path: Path,
) -> None:
    """Visualize global overlap metrics across top-k configs.

    Args:
        global_df: Global overlap dataframe.
        output_path: PNG file path.

    Returns:
        None. Figure is saved to disk.
    """

    if global_df.empty:
        raise ValueError("global_df is empty; cannot visualize global overlap metrics.")

    plot_df = global_df.copy()
    plot_df = plot_df.sort_values(["mode", "top_pct", "config_name"]).reset_index(drop=True)

    metric_keys = [
        "recall_over_unique",
        "precision_over_topk",
        "jaccard",
        "overlap_ratio_over_all",
    ]

    melted = plot_df.melt(
        id_vars=["config_name"],
        value_vars=metric_keys,
        var_name="metric",
        value_name="value",
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    sns.barplot(data=melted, x="metric", y="value", hue="config_name", ax=axes[0])
    axes[0].set_title("Global Overlap Metrics")
    axes[0].set_xlabel("Metric")
    axes[0].set_ylabel("Ratio")

    ratio_ylim = compute_tight_ratio_ylim(
        series_list=[melted["value"].to_numpy(dtype=np.float64)],
        pad_ratio=0.10,
        min_span=0.10,
    )
    axes[0].set_ylim(*ratio_ylim)
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].legend(loc="best")

    sns.barplot(data=plot_df, x="config_name", y="enrichment_over_random", ax=axes[1], color="#4c72b0")
    axes[1].set_title("Global Enrichment over Random Baseline")
    axes[1].set_xlabel("Config")
    axes[1].set_ylabel("Enrichment")
    axes[1].tick_params(axis="x", rotation=20)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved global overlap figure: {output_path}")


def visualize_if_unique_topk_distribution(
    sampled_abs_delta_by_config: Mapping[str, Mapping[str, List[float]]],
    output_path: Path,
) -> None:
    """Visualize abs-delta distributions for overlap relation groups per config.

    Args:
        sampled_abs_delta_by_config: Nested mapping
            `config_name -> {overlap, unique_only, topk_only} -> samples`.
        output_path: PNG file path.

    Returns:
        None. Figure is saved to disk.
    """

    config_names = sorted(sampled_abs_delta_by_config.keys())
    if not config_names:
        raise ValueError("No configs found for distribution visualization.")

    n_configs = len(config_names)
    n_cols = 2
    n_rows = int(np.ceil(n_configs / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows), constrained_layout=True)
    axes_array = np.atleast_1d(axes).reshape(n_rows, n_cols)

    for config_index, config_name in enumerate(config_names):
        row = config_index // n_cols
        col = config_index % n_cols
        axis = axes_array[row, col]

        group_samples = sampled_abs_delta_by_config[config_name]
        plotted_any = False

        for group_name, values in group_samples.items():
            if not values:
                continue

            values_array = np.asarray(values, dtype=np.float64)
            log_values = np.log10(np.maximum(values_array, 1e-20))

            sns.histplot(
                log_values,
                bins=90,
                stat="density",
                element="step",
                fill=False,
                linewidth=1.4,
                label=f"{group_name} (n={len(values)})",
                ax=axis,
            )
            plotted_any = True

        axis.set_title(f"{config_name}: log10(abs(Delta_if)) distribution")
        axis.set_xlabel("log10(abs(Delta_if))")
        axis.set_ylabel("Density")

        if plotted_any:
            axis.legend(loc="upper left")
        else:
            axis.text(
                0.5,
                0.5,
                "No sampled values",
                transform=axis.transAxes,
                ha="center",
                va="center",
            )

    # Hide unused axes when config count is odd.
    for remaining_index in range(n_configs, n_rows * n_cols):
        row = remaining_index // n_cols
        col = remaining_index % n_cols
        axes_array[row, col].axis("off")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.show()
    print(f"Saved distribution overlap figure: {output_path}")


if "task_vectors" not in globals():
    raise RuntimeError(
        "`task_vectors` is not available in memory. "
        "Re-run the task-vector extraction cell before running this analysis section."
    )

if "if" not in task_vectors or "math" not in task_vectors:
    raise KeyError("task_vectors must include both 'if' and 'math' keys.")

if_unique_topk_layerwise_df, if_unique_topk_global_df, if_unique_topk_samples = compute_if_unique_vs_topk_overlap(
    if_task_vector=task_vectors["if"],
    math_task_vector=task_vectors["math"],
    threshold=float(RUNTIME.threshold),
    config_list=TOPK_COMPARISON_CONFIGS,
    sample_cap_per_group=int(TOPK_DISTRIBUTION_SAMPLE_CAP_PER_GROUP),
    sample_seed=int(RUNTIME.seed),
)

if_unique_topk_layerwise_df.to_csv(IF_UNIQUE_TOPK_LAYERWISE_CSV, index=False)
if_unique_topk_global_df.to_csv(IF_UNIQUE_TOPK_GLOBAL_CSV, index=False)

save_json(
    {
        "base_model_id": RUNTIME.base_model_id,
        "if_model_path": str(RUNTIME.if_model_path),
        "math_model_path": str(RUNTIME.math_model_path),
        "threshold": float(RUNTIME.threshold),
        "topk_configs": TOPK_COMPARISON_CONFIGS,
        "global_metrics": if_unique_topk_global_df.to_dict(orient="records"),
    },
    IF_UNIQUE_TOPK_GLOBAL_JSON,
)

visualize_if_unique_topk_layerwise(
    layerwise_df=if_unique_topk_layerwise_df,
    output_path=IF_UNIQUE_TOPK_LAYERWISE_PNG,
)
visualize_if_unique_topk_global(
    global_df=if_unique_topk_global_df,
    output_path=IF_UNIQUE_TOPK_GLOBAL_PNG,
)
visualize_if_unique_topk_distribution(
    sampled_abs_delta_by_config=if_unique_topk_samples,
    output_path=IF_UNIQUE_TOPK_DIST_PNG,
)

print(f"Saved layer-wise CSV: {IF_UNIQUE_TOPK_LAYERWISE_CSV}")
print(f"Saved global CSV: {IF_UNIQUE_TOPK_GLOBAL_CSV}")
print(f"Saved global JSON: {IF_UNIQUE_TOPK_GLOBAL_JSON}")

print(f"Total layer-wise rows: {len(if_unique_topk_layerwise_df)}")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(if_unique_topk_layerwise_df)
display(if_unique_topk_global_df)


## FP32 Model Saving Extensions (Requested)

The following cells implement the newly requested actions:
1. Save IF-only and Math-only **unique-vector reconstruction** models (`base + unique_delta * 1.0`).
2. Save FP32 **RAM** merged model.
3. Save FP32 **RAM+ (`arm-r-v2`)** merged model.

All load/merge/save operations in these cells are explicitly FP32.


In [ ]:
MERGE_OUTPUT_ROOT = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ram-fp32"
)
RAM_PLUS_RESCALE_FACTOR = 1.1
UNIQUE_VECTOR_SCALE = 1.0

MERGE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)



def build_unique_task_vectors_two_tasks(
    if_task_vector: Mapping[str, torch.Tensor],
    math_task_vector: Mapping[str, torch.Tensor],
    threshold: float,
) -> tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor], Dict[str, Any]]:
    """Construct IF-unique and Math-unique vectors from two task vectors.

    Definitions:
    - `m_if = |Delta_if| > threshold`
    - `m_math = |Delta_math| > threshold`
    - `unique_if = m_if & ~m_math`
    - `unique_math = m_math & ~m_if`

    Args:
        if_task_vector: IF task-vector mapping (`Delta_if`).
        math_task_vector: Math task-vector mapping (`Delta_math`).
        threshold: Activity threshold used for unique-mask construction.

    Returns:
        Tuple of:
        - IF-unique task vector dictionary
        - Math-unique task vector dictionary
        - Summary dictionary with global unique counts and ratios
    """

    common_keys = sorted(set(if_task_vector.keys()) & set(math_task_vector.keys()))
    if not common_keys:
        raise ValueError("No common keys for unique-vector extraction.")

    unique_if_vector: Dict[str, torch.Tensor] = {}
    unique_math_vector: Dict[str, torch.Tensor] = {}

    total_numel = 0
    total_if_unique = 0
    total_math_unique = 0

    with torch.no_grad():
        for parameter_name in tqdm(common_keys, desc="Build unique-only vectors"):
            if_delta = if_task_vector[parameter_name].detach().to(torch.float32)
            math_delta = math_task_vector[parameter_name].detach().to(torch.float32)

            if if_delta.shape != math_delta.shape:
                raise ValueError(
                    "Shape mismatch during unique-vector extraction. "
                    f"parameter={parameter_name}"
                )

            if_mask = torch.abs(if_delta) > threshold
            math_mask = torch.abs(math_delta) > threshold

            unique_if_mask = if_mask & (~math_mask)
            unique_math_mask = math_mask & (~if_mask)

            # Preserve signed delta values only on unique coordinates.
            unique_if_vector[parameter_name] = torch.where(
                unique_if_mask,
                if_delta,
                torch.zeros_like(if_delta),
            )
            unique_math_vector[parameter_name] = torch.where(
                unique_math_mask,
                math_delta,
                torch.zeros_like(math_delta),
            )

            total_numel += int(if_delta.numel())
            total_if_unique += int(unique_if_mask.sum().item())
            total_math_unique += int(unique_math_mask.sum().item())

    summary = {
        "threshold": float(threshold),
        "num_elements": int(total_numel),
        "if_unique_elements": int(total_if_unique),
        "math_unique_elements": int(total_math_unique),
        "if_unique_ratio_over_all": safe_divide(float(total_if_unique), float(total_numel)),
        "math_unique_ratio_over_all": safe_divide(float(total_math_unique), float(total_numel)),
    }
    return unique_if_vector, unique_math_vector, summary



def apply_single_task_vector_update_inplace(
    base_model: AutoModelForCausalLM,
    task_vector: Mapping[str, torch.Tensor],
    scale: float,
) -> Dict[str, Any]:
    """Apply `theta <- theta + scale * Delta` in-place in FP32.

    Args:
        base_model: Model to update in-place.
        task_vector: Task-vector mapping aligned to model parameter keys.
        scale: Scalar multiplier applied to each delta tensor.

    Returns:
        Update summary dictionary.
    """

    base_params = dict(base_model.named_parameters())
    vector_keys = set(task_vector.keys())
    param_keys = set(base_params.keys())
    if vector_keys != param_keys:
        raise ValueError(
            "Task-vector keys must exactly match model parameter keys. "
            f"missing_in_vector={sorted(param_keys - vector_keys)[:5]}, "
            f"missing_in_model={sorted(vector_keys - param_keys)[:5]}"
        )

    updated_parameter_count = 0
    skipped_non_floating_count = 0

    with torch.no_grad():
        for parameter_name, parameter in tqdm(base_params.items(), desc="Apply single-vector update"):
            if not torch.is_floating_point(parameter.data):
                skipped_non_floating_count += 1
                continue

            base_fp32 = parameter.data.detach().to(torch.float32)
            delta_fp32 = task_vector[parameter_name].to(torch.float32)
            updated = base_fp32 + (float(scale) * delta_fp32)
            parameter.data.copy_(updated)
            updated_parameter_count += 1

    total_l1 = float(sum(tensor.abs().sum().item() for tensor in task_vector.values()))
    return {
        "updated_parameter_count": int(updated_parameter_count),
        "skipped_non_floating_count": int(skipped_non_floating_count),
        "scale": float(scale),
        "task_vector_total_l1_norm": total_l1,
    }



def apply_ram_merge_inplace(
    base_model: AutoModelForCausalLM,
    task_vector_list: List[Mapping[str, torch.Tensor]],
    threshold: float,
) -> Dict[str, Any]:
    """Apply RAM baseline merge in-place (active-coordinate average).

    This implementation follows the provided public `agentic_reinforcement_merge`
    behavior while enforcing FP32 arithmetic.

    Args:
        base_model: Base model to overwrite with RAM merged weights.
        task_vector_list: List of task-vector mappings.
        threshold: Activity threshold.

    Returns:
        Merge summary dictionary.
    """

    if len(task_vector_list) < 1:
        raise ValueError("task_vector_list must contain at least one task vector.")

    base_params = dict(base_model.named_parameters())
    common_keys = set(base_params.keys())
    for task_vector in task_vector_list:
        common_keys &= set(task_vector.keys())

    if common_keys != set(base_params.keys()):
        raise ValueError("Task vectors must cover all model parameter keys for RAM merge.")

    updated_parameter_count = 0
    skipped_non_floating_count = 0

    with torch.no_grad():
        for parameter_name, parameter in tqdm(base_params.items(), desc="Apply RAM merge"):
            if not torch.is_floating_point(parameter.data):
                skipped_non_floating_count += 1
                continue

            diffs = torch.stack(
                [task_vector[parameter_name].to(torch.float32) for task_vector in task_vector_list],
                dim=0,
            )
            change_mask = (torch.abs(diffs) > float(threshold)).to(torch.float32)

            sum_diff = (diffs * change_mask).sum(dim=0)
            count = change_mask.sum(dim=0)
            avg_diff = sum_diff / torch.clamp(count, min=1.0)
            diff_final = torch.where(count > 0, avg_diff, torch.zeros_like(avg_diff))

            base_fp32 = parameter.data.detach().to(torch.float32)
            parameter.data.copy_(base_fp32 + diff_final)
            updated_parameter_count += 1

    return {
        "updated_parameter_count": int(updated_parameter_count),
        "skipped_non_floating_count": int(skipped_non_floating_count),
        "threshold": float(threshold),
        "num_tasks": int(len(task_vector_list)),
        "method": "ram_arm",
    }



def compute_ram_plus_rescales_v2(
    task_vector_list: List[Mapping[str, torch.Tensor]],
    threshold: float,
    r: float,
) -> Dict[str, Any]:
    """Compute per-task rescale factors for RAM+ (`arm-r-v2`).

    Public-code ratio definition:
    - `ratio_j = overlap_j / (changed_j - overlap_j)`

    Stability notes:
    - If `changed_j == 0`, ratio is treated as `0.0`.
    - If `(changed_j - overlap_j) <= 0`, ratio saturates to `1.0` to match
      the clipped behavior of the original formula under infinite ratio.

    Args:
        task_vector_list: List of task-vector mappings.
        threshold: Activity threshold.
        r: Rescale factor hyperparameter.

    Returns:
        Dictionary containing changed counts, overlap counts, ratios, and rescales.
    """

    if len(task_vector_list) < 1:
        raise ValueError("task_vector_list must contain at least one task vector.")

    r_value = float(r)
    if r_value <= 1.0:
        r_value = 1.0

    common_keys = set(task_vector_list[0].keys())
    for task_vector in task_vector_list[1:]:
        common_keys &= set(task_vector.keys())
    common_keys = sorted(common_keys)

    num_tasks = len(task_vector_list)
    changed_counts = [0 for _ in range(num_tasks)]
    overlap_counts = [0 for _ in range(num_tasks)]

    with torch.no_grad():
        for parameter_name in tqdm(common_keys, desc="RAM+ v2 overlap stats"):
            diffs = torch.stack(
                [task_vector[parameter_name].to(torch.float32) for task_vector in task_vector_list],
                dim=0,
            )
            change_mask = torch.abs(diffs) > float(threshold)
            overlap_any = change_mask.sum(dim=0) >= 2

            change_flat = change_mask.view(num_tasks, -1)
            overlap_flat = overlap_any.view(-1)

            for task_index in range(num_tasks):
                active_flat = change_flat[task_index]
                changed_counts[task_index] += int(active_flat.sum().item())
                overlap_counts[task_index] += int((active_flat & overlap_flat).sum().item())

    overlap_ratios: List[float] = []
    rescales: List[float] = []
    for task_index in range(num_tasks):
        changed = float(changed_counts[task_index])
        overlap = float(overlap_counts[task_index])

        if changed <= 0.0:
            ratio = 0.0
        else:
            non_overlap = changed - overlap
            if non_overlap <= 0.0:
                ratio = 1.0
            else:
                ratio = overlap / non_overlap

        overlap_ratios.append(float(ratio))

        clipped = min(2.0, min(1.0, float(ratio)))
        rescale = 1.0 + (r_value - 1.0) * clipped
        rescales.append(float(rescale))

    return {
        "changed_counts": [int(value) for value in changed_counts],
        "overlap_counts": [int(value) for value in overlap_counts],
        "overlap_ratios": overlap_ratios,
        "rescales": rescales,
        "r_effective": float(r_value),
    }



def apply_ram_plus_merge_v2_inplace(
    base_model: AutoModelForCausalLM,
    task_vector_list: List[Mapping[str, torch.Tensor]],
    threshold: float,
    r: float,
) -> Dict[str, Any]:
    """Apply RAM+ (`arm-r-v2`) merge in-place in FP32.

    Behavior matches the provided public `agentic_reinforcement_merge_rescale_v2`:
    - overlap coordinates (`count >= 2`): use active average
    - non-overlap coordinates (`count == 1`): use weighted sum with per-task rescale

    Args:
        base_model: Base model to overwrite with RAM+ merged weights.
        task_vector_list: List of task vectors.
        threshold: Activity threshold.
        r: RAM+ rescale factor.

    Returns:
        Merge summary dictionary containing overlap stats and update counters.
    """

    stats = compute_ram_plus_rescales_v2(
        task_vector_list=task_vector_list,
        threshold=threshold,
        r=r,
    )

    rescales = stats["rescales"]
    num_tasks = len(task_vector_list)

    base_params = dict(base_model.named_parameters())
    common_keys = set(base_params.keys())
    for task_vector in task_vector_list:
        common_keys &= set(task_vector.keys())

    if common_keys != set(base_params.keys()):
        raise ValueError("Task vectors must cover all model parameter keys for RAM+ merge.")

    updated_parameter_count = 0
    skipped_non_floating_count = 0

    with torch.no_grad():
        for parameter_name, parameter in tqdm(base_params.items(), desc="Apply RAM+ v2 merge"):
            if not torch.is_floating_point(parameter.data):
                skipped_non_floating_count += 1
                continue

            diffs = torch.stack(
                [task_vector[parameter_name].to(torch.float32) for task_vector in task_vector_list],
                dim=0,
            )

            change_mask = (torch.abs(diffs) > float(threshold)).to(torch.float32)
            sum_diff = (diffs * change_mask).sum(dim=0)
            count = change_mask.sum(dim=0)
            avg_diff = sum_diff / torch.clamp(count, min=1.0)

            overlap_mask = count >= 2
            non_overlap_mask = count == 1

            rescales_tensor = torch.tensor(
                rescales,
                dtype=diffs.dtype,
                device=diffs.device,
            ).view((num_tasks,) + (1,) * (diffs.dim() - 1))
            weighted_sum = (diffs * change_mask * rescales_tensor).sum(dim=0)

            diff_final = torch.zeros_like(avg_diff)
            diff_final = torch.where(overlap_mask, avg_diff, diff_final)
            diff_final = torch.where(non_overlap_mask, weighted_sum, diff_final)

            base_fp32 = parameter.data.detach().to(torch.float32)
            parameter.data.copy_(base_fp32 + diff_final)
            updated_parameter_count += 1

    return {
        "updated_parameter_count": int(updated_parameter_count),
        "skipped_non_floating_count": int(skipped_non_floating_count),
        "threshold": float(threshold),
        "num_tasks": int(num_tasks),
        "method": "ram_plus_arm_r_v2",
        "ram_plus_stats": stats,
    }



def save_fp32_model_and_metadata(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save FP32 model/tokenizer and metadata artifact.

    Args:
        model: Model instance expected to hold FP32 parameters.
        tokenizer: Tokenizer to save with the checkpoint.
        output_dir: Destination directory.
        metadata: JSON-serializable metadata payload.

    Returns:
        None. Artifacts are written to disk.
    """

    assert_model_float32(model, model_label=f"to_save:{output_dir.name}")
    output_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(dict(metadata), output_dir / "merge_metadata.json")


In [ ]:
# ----------------------------------------------------------------------------------
# Execute requested FP32 saves:
# 1) IF unique-only reconstruction
# 2) Math unique-only reconstruction
# 3) RAM merged model
# 4) RAM+ (arm-r-v2) merged model
# ----------------------------------------------------------------------------------

# Release large base model from earlier diagnostic steps before reloading base models
# for each save target. This reduces peak host memory pressure.
if "base_model" in globals():
    del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

threshold = float(RUNTIME.threshold)

unique_if_vector, unique_math_vector, unique_summary = build_unique_task_vectors_two_tasks(
    if_task_vector=task_vectors["if"],
    math_task_vector=task_vectors["math"],
    threshold=threshold,
)
print("Unique vector summary:")
print(json.dumps(unique_summary, indent=2, ensure_ascii=False))

saved_model_dirs: Dict[str, str] = {}

# ----------------------------------------
# Unique IF-only model: base + unique_if * 1
# ----------------------------------------
print("
[Save] Building unique IF-only reconstruction model (FP32)...")
unique_if_model, unique_if_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(unique_if_model, model_label="unique_if_base_reload")

unique_if_summary = apply_single_task_vector_update_inplace(
    base_model=unique_if_model,
    task_vector=unique_if_vector,
    scale=float(UNIQUE_VECTOR_SCALE),
)
unique_if_output_dir = MERGE_OUTPUT_ROOT / f"unique_if_only_scale_{UNIQUE_VECTOR_SCALE:g}_fp32"

save_fp32_model_and_metadata(
    model=unique_if_model,
    tokenizer=unique_if_tokenizer,
    output_dir=unique_if_output_dir,
    metadata={
        "method": "unique_if_only_reconstruction",
        "formula": "theta = theta_base + 1.0 * Delta_if_unique",
        "scale": float(UNIQUE_VECTOR_SCALE),
        "threshold": threshold,
        "base_model_id": RUNTIME.base_model_id,
        "if_model_path": str(RUNTIME.if_model_path),
        "math_model_path": str(RUNTIME.math_model_path),
        "unique_summary": unique_summary,
        "update_summary": unique_if_summary,
    },
)
saved_model_dirs["unique_if_only"] = str(unique_if_output_dir)

# Cleanup before next model reconstruction.
del unique_if_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------
# Unique Math-only model: base + unique_math * 1
# ------------------------------------------
print("
[Save] Building unique Math-only reconstruction model (FP32)...")
unique_math_model, unique_math_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(unique_math_model, model_label="unique_math_base_reload")

unique_math_summary = apply_single_task_vector_update_inplace(
    base_model=unique_math_model,
    task_vector=unique_math_vector,
    scale=float(UNIQUE_VECTOR_SCALE),
)
unique_math_output_dir = MERGE_OUTPUT_ROOT / f"unique_math_only_scale_{UNIQUE_VECTOR_SCALE:g}_fp32"

save_fp32_model_and_metadata(
    model=unique_math_model,
    tokenizer=unique_math_tokenizer,
    output_dir=unique_math_output_dir,
    metadata={
        "method": "unique_math_only_reconstruction",
        "formula": "theta = theta_base + 1.0 * Delta_math_unique",
        "scale": float(UNIQUE_VECTOR_SCALE),
        "threshold": threshold,
        "base_model_id": RUNTIME.base_model_id,
        "if_model_path": str(RUNTIME.if_model_path),
        "math_model_path": str(RUNTIME.math_model_path),
        "unique_summary": unique_summary,
        "update_summary": unique_math_summary,
    },
)
saved_model_dirs["unique_math_only"] = str(unique_math_output_dir)

# Cleanup before RAM merge reconstruction.
del unique_math_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------
# RAM merged model (FP32)
# ------------------------
print("
[Save] Building RAM merged model (FP32)...")
ram_model, ram_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(ram_model, model_label="ram_base_reload")

ram_summary = apply_ram_merge_inplace(
    base_model=ram_model,
    task_vector_list=[task_vectors["if"], task_vectors["math"]],
    threshold=threshold,
)
ram_output_dir = MERGE_OUTPUT_ROOT / "ram_arm_fp32"

save_fp32_model_and_metadata(
    model=ram_model,
    tokenizer=ram_tokenizer,
    output_dir=ram_output_dir,
    metadata={
        "method": "ram_arm",
        "formula": "theta = theta_base + avg_active(Delta_if, Delta_math)",
        "threshold": threshold,
        "base_model_id": RUNTIME.base_model_id,
        "if_model_path": str(RUNTIME.if_model_path),
        "math_model_path": str(RUNTIME.math_model_path),
        "update_summary": ram_summary,
    },
)
saved_model_dirs["ram"] = str(ram_output_dir)

# Cleanup before RAM+ merge reconstruction.
del ram_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -----------------------------
# RAM+ merged model (arm-r-v2)
# -----------------------------
print("
[Save] Building RAM+ (arm-r-v2) merged model (FP32)...")
ram_plus_model, ram_plus_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(ram_plus_model, model_label="ram_plus_base_reload")

ram_plus_summary = apply_ram_plus_merge_v2_inplace(
    base_model=ram_plus_model,
    task_vector_list=[task_vectors["if"], task_vectors["math"]],
    threshold=threshold,
    r=float(RAM_PLUS_RESCALE_FACTOR),
)
ram_plus_output_dir = MERGE_OUTPUT_ROOT / f"ram_plus_arm_r_v2_r{RAM_PLUS_RESCALE_FACTOR:g}_fp32"

save_fp32_model_and_metadata(
    model=ram_plus_model,
    tokenizer=ram_plus_tokenizer,
    output_dir=ram_plus_output_dir,
    metadata={
        "method": "ram_plus_arm_r_v2",
        "formula": "overlap->avg_active, non_overlap->rescaled_weighted_sum",
        "threshold": threshold,
        "rescale_factor_r": float(RAM_PLUS_RESCALE_FACTOR),
        "base_model_id": RUNTIME.base_model_id,
        "if_model_path": str(RUNTIME.if_model_path),
        "math_model_path": str(RUNTIME.math_model_path),
        "update_summary": ram_plus_summary,
    },
)
saved_model_dirs["ram_plus"] = str(ram_plus_output_dir)

# Final cleanup for long-running notebook sessions.
del ram_plus_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("
Saved model directories (FP32):")
for key, value in saved_model_dirs.items():
    print(f"- {key}: {value}")


In [ ]:
# -----------------------------------------------------------------
# Post-save smoke checks: directory existence + quick metadata check
# -----------------------------------------------------------------

expected_saved_dirs = [
    MERGE_OUTPUT_ROOT / f"unique_if_only_scale_{UNIQUE_VECTOR_SCALE:g}_fp32",
    MERGE_OUTPUT_ROOT / f"unique_math_only_scale_{UNIQUE_VECTOR_SCALE:g}_fp32",
    MERGE_OUTPUT_ROOT / "ram_arm_fp32",
    MERGE_OUTPUT_ROOT / f"ram_plus_arm_r_v2_r{RAM_PLUS_RESCALE_FACTOR:g}_fp32",
]

for output_dir in expected_saved_dirs:
    if not output_dir.exists():
        raise FileNotFoundError(f"Saved model directory not found: {output_dir}")
    metadata_path = output_dir / "merge_metadata.json"
    if not metadata_path.exists():
        raise FileNotFoundError(f"merge_metadata.json not found: {metadata_path}")

print("All requested FP32 model save targets exist.")
for output_dir in expected_saved_dirs:
    print(f"- {output_dir}")

# Optional cleanup now that all requested operations are complete.
if "task_vectors" in globals():
    del task_vectors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
